# 🌙 02. Lunar Surface Terrain Geomorphology & Crater Density Analysis

**Mission Context**: Automated geological characterization of lunar surface features for landing site selection and spatial feature landmarking.  
**Objectives**:
- Detect impact craters across multiple scales (Hough Transform & Laplacian of Gaussian).
- Extract linear structures: ridges (wrinkle ridges) and valleys (sinuous rilles).
- Segment permanent & grazing shadows.
- Compute Crater Spatial Density ($N / \text{km}^2$) and Grey-Level Co-occurrence Matrix (GLCM) Texture Complexity.
- Export `terrain_report.csv` and `terrain_statistics.json` with high-resolution PNG maps.


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import json

sys.path.append(str(Path.cwd().parent))
from lunar_core.config import load_config
from lunar_core.data_loader import LunarImageLoader
from lunar_core.terrain import LunarTerrainAnalyzer
from lunar_core.synthetic_data import LunarSyntheticGenerator

config = load_config()


In [ ]:
# Load sample lunar image
img_files = list(Path("data").glob("*/*.png"))
if not img_files:
    gen = LunarSyntheticGenerator(size=(512, 512), seed=42)
    gen.populate_sample_datasets("data")
    img_files = list(Path("data").glob("*/*.png"))

img_path = str(img_files[0])
img_gray = LunarImageLoader.load_image(img_path)
print(f"Loaded {img_path} with shape: {img_gray.shape}")


In [ ]:
# Initialize and execute terrain analysis
analyzer = LunarTerrainAnalyzer(crater_min_r=6, crater_max_r=90, shadow_threshold=40)
terrain_res = analyzer.analyze(img_gray)

print("--- Terrain Geomorphology Results ---")
print(f"Detected Craters: {terrain_res['num_craters_detected']}")
print(f"Crater Density: {terrain_res['crater_density']} craters/km²")
print(f"Shadow Coverage: {terrain_res['shadow_percentage']}%")
print(f"Texture Complexity Index: {terrain_res['texture_complexity']}")


In [ ]:
# High-Resolution Visualization Suite
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. Original Image with Crater Boundaries
annotated = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2BGR)
for cx, cy, r in terrain_res['craters_list']:
    cv2.circle(annotated, (cx, cy), r, (0, 255, 0), 2)
    cv2.circle(annotated, (cx, cy), 3, (0, 0, 255), -1)

axes[0, 0].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title(f"Crater Extractions (N={terrain_res['num_craters_detected']})", fontweight='bold')
axes[0, 0].axis('off')

# 2. Crater Density Heatmap
im_dens = axes[0, 1].imshow(terrain_res['crater_density_map'], cmap='hot')
axes[0, 1].set_title("Crater Spatial Density Map", fontweight='bold')
plt.colorbar(im_dens, ax=axes[0, 1], fraction=0.046)
axes[0, 1].axis('off')

# 3. Binary Shadow Segmentation
axes[0, 2].imshow(terrain_res['shadow_mask'], cmap='Blues_r')
axes[0, 2].set_title(f"Shadow Mask ({terrain_res['shadow_percentage']}%)", fontweight='bold')
axes[0, 2].axis('off')

# 4. Structural Ridges (Top-Hat)
axes[1, 0].imshow(terrain_res['ridges_map'], cmap='magma')
axes[1, 0].set_title("Wrinkle Ridge Lineaments", fontweight='bold')
axes[1, 0].axis('off')

# 5. Valleys & Rilles (Black-Hat)
axes[1, 1].imshow(terrain_res['valleys_map'], cmap='viridis')
axes[1, 1].set_title("Sinuous Rilles & Valleys", fontweight='bold')
axes[1, 1].axis('off')

# 6. Combined Geomorphology Overlay
overlay = cv2.addWeighted(img_gray, 0.7, (terrain_res['crater_density_map']*255).astype(np.uint8), 0.3, 0)
axes[1, 2].imshow(overlay, cmap='copper')
axes[1, 2].set_title(f"Terrain Complexity: {terrain_res['texture_complexity']}", fontweight='bold')
axes[1, 2].axis('off')

plt.tight_layout()
os.makedirs("outputs/visualizations", exist_ok=True)
plt.savefig("outputs/visualizations/02_terrain_geomorphology.png", dpi=300)
plt.show()


In [ ]:
# Export Reports: terrain_report.csv & terrain_statistics.json
records = [{
    "image_path": img_path,
    "num_craters": terrain_res["num_craters_detected"],
    "crater_density_per_sq_km": terrain_res["crater_density"],
    "shadow_percentage": terrain_res["shadow_percentage"],
    "texture_complexity": terrain_res["texture_complexity"]
}]

df_terrain = pd.DataFrame(records)
df_terrain.to_csv("outputs/reports/terrain_report.csv", index=False)

stats_dict = {
    "num_craters_detected": terrain_res["num_craters_detected"],
    "crater_density": terrain_res["crater_density"],
    "shadow_percentage": terrain_res["shadow_percentage"],
    "texture_complexity": terrain_res["texture_complexity"]
}
with open("outputs/reports/terrain_statistics.json", "w") as f:
    json.dump(stats_dict, f, indent=4)

print("Exported outputs/reports/terrain_report.csv and terrain_statistics.json")
